In [1]:
!pip install yfinance pandas numpy

# Income Objective Portfolio Construction (Bursa Malaysia)

Phase 1:
- Data collection
- Data cleaning
- Return computation
- Volatility classification (Stage 1 & Stage 2)

Benchmark: FTSE Bursa Malaysia KLCI

In [2]:
import yfinance as yf
import pandas as pd
import numpy as np
import os

## 1. Candidate Pool Selection

A diversified pool of Bursa Malaysia stocks including mix of sectors (banking, utilities, tech, industrials, etc.) 
- and different market capitalisations (large, mid, small).
- This avoids selection bias and allows volatility classification to emerge from data.


In [3]:
stocks = [
    "1155.KL","1295.KL","1023.KL","5347.KL","5225.KL",
    "5819.KL","1961.KL","2445.KL","3816.KL","6012.KL",
    "4863.KL","6888.KL","1082.KL","5183.KL","6033.KL",
    "1818.KL","4707.KL","4197.KL","3182.KL","4715.KL",
    "1066.KL","2488.KL","4677.KL","6742.KL","1562.KL",
    "5296.KL","7113.KL","7106.KL","5168.KL","6947.KL",
    "0138.KL","5216.KL","7277.KL","7084.KL","7203.KL",
    "5285.KL","5681.KL","1619.KL","5211.KL","0166.KL"
]

# Benchmark index for beta calculation later
benchmark = "^KLSE"  # FTSE Bursa Malaysia KLCI

## 2. Data Collection

Download 2 years of daily closing prices using Yahoo Finance.

In [4]:
print("Downloading data...")
data = yf.download(stocks + [benchmark], period="2y", auto_adjust=True, progress=True)

# Extract closing prices (standard for return calculations)
prices = data["Close"]

# Separate benchmark and stock prices (important for beta calculation later)
benchmark_prices = prices[benchmark]
stock_prices = prices.drop(columns=[benchmark])

[*********************100%***********************]  41 of 41 completed


## 3. Data Cleaning

- Allow up to 10% missing values to maintain diversification.
- This improves diversification while maintaining acceptable data quality
- Strict removal (0% missing) would reduce the investment universe too much


In [5]:
stock_prices = stock_prices.dropna(axis=1, thresh=len(stock_prices) * 0.9)

print(f"\nRemaining stocks after cleaning: {len(stock_prices.columns)}")
print(stock_prices.columns.tolist())


Remaining stocks after cleaning: 40
['0138.KL', '0166.KL', '1023.KL', '1066.KL', '1082.KL', '1155.KL', '1295.KL', '1562.KL', '1619.KL', '1818.KL', '1961.KL', '2445.KL', '2488.KL', '3182.KL', '3816.KL', '4197.KL', '4677.KL', '4707.KL', '4715.KL', '4863.KL', '5168.KL', '5183.KL', '5211.KL', '5216.KL', '5225.KL', '5285.KL', '5296.KL', '5347.KL', '5681.KL', '5819.KL', '6012.KL', '6033.KL', '6742.KL', '6888.KL', '6947.KL', '7084.KL', '7106.KL', '7113.KL', '7203.KL', '7277.KL']


## 4. Return Calculation

- Compute daily percentage returns.
- Convert prices into percentage returns
- This standardises performance across different stock price levels

In [6]:
returns = stock_prices.pct_change().dropna()
benchmark_returns = benchmark_prices.pct_change().dropna()

C:\Users\user\AppData\Local\Temp\ipykernel_10360\552055115.py:2: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  benchmark_returns = benchmark_prices.pct_change().dropna()


## 5. Statistical Measures
- Mean Return = expected daily return
- Volatility = standard deviation of daily returns (risk)

In [7]:
mean_returns = returns.mean()
volatility = returns.std()

stats_full = pd.DataFrame({
    "Mean Return": mean_returns,
    "Volatility": volatility,
    "Annual Return": mean_returns * 252,    # annualised return, 252 trading days
    "Annual Risk": volatility * np.sqrt(252)# annualised risk
})

# Remove zero volatility (avoid division errors)
stats_full = stats_full[stats_full["Volatility"] > 0]

# Return/Risk Ratio measures efficiency (return per unit of risk)
stats_full["Return/Risk Ratio"] = stats_full["Mean Return"] / stats_full["Volatility"]


## 6. Stage 1: Low Volatility Stocks
- Income objective → select only stocks with positive returns
- Negative-return stocks are excluded as they contradict income generation goals


In [8]:
stats_positive = stats_full[stats_full["Mean Return"] > 0].copy()

print(f"\nStocks with positive returns: {len(stats_positive)}")
print(f"Stocks excluded (negative returns): {len(stats_full) - len(stats_positive)}")

# Select top 10 stocks with highest return-to-risk ratio
# This proxies stable and efficient income-generating stocks

# Step 1: Try assignment condition
low_vol_condition = stats_positive[
    stats_positive["Mean Return"] > stats_positive["Volatility"]
]

# Step 2: Apply logic
if len(low_vol_condition) >= 10:
    low_vol = low_vol_condition.head(10)
    method_used = "Assignment condition: Mean return > Volatility"
else:
    print("⚠️ No stocks met strict condition → using lowest volatility fallback")
    low_vol = stats_positive.nsmallest(10, "Volatility")
    method_used = "Fallback: Lowest volatility selection"


Stocks with positive returns: 27
Stocks excluded (negative returns): 13
⚠️ No stocks met strict condition → using lowest volatility fallback


## 7. Stage 2: High Volatility Stocks

- Use FULL universe (including negative returns)
- Mandate defines high volatility as:
- volatility > 3 × mean return

In [9]:
high_vol_candidates = stats_full[
    stats_full["Volatility"] > 3 * stats_full["Mean Return"]
]

print(f"\nStocks meeting high-vol condition: {len(high_vol_candidates)}")

# Select top 5 highest volatility stocks
high_vol = high_vol_candidates.sort_values(
    by="Volatility", ascending=False
).head(5)

# Fallback if insufficient stocks meet condition
if len(high_vol) < 5:
    print("⚠️ Not enough stocks met high-vol condition. Using highest volatility stocks instead.")
    high_vol = stats_full.sort_values(
        by="Volatility", ascending=False
    ).head(5)

# Flag whether each stock strictly meets condition
high_vol["Meets_Condition"] = high_vol["Volatility"] > 3 * high_vol["Mean Return"]



Stocks meeting high-vol condition: 40


## 8. Ouput Results

In [10]:
print("\n" + "="*70)
print("LOW VOLATILITY STOCKS (Stage 1)")
print("="*70)
print(low_vol[["Annual Return", "Annual Risk", "Return/Risk Ratio"]])

print("\n" + "="*70)
print("HIGH VOLATILITY STOCKS (Stage 2 Addition)")
print("="*70)
print(high_vol[["Annual Return", "Annual Risk", "Volatility", "Mean Return", "Meets_Condition"]])

# Highlight high-vol stocks with negative returns (important insight)
negative_high_vol = high_vol[high_vol["Mean Return"] < 0]
if len(negative_high_vol) > 0:
    print("\n⚠️ High-volatility stocks with NEGATIVE returns:")
    print(negative_high_vol.index.tolist())


LOW VOLATILITY STOCKS (Stage 1)
         Annual Return  Annual Risk  Return/Risk Ratio
Ticker                                                
1155.KL       0.136381     0.139714           0.061491
1066.KL       0.288231     0.161996           0.112082
5819.KL       0.128188     0.165963           0.048656
5225.KL       0.192771     0.169497           0.071644
1082.KL       0.117013     0.169923           0.043379
6033.KL       0.057347     0.171179           0.021104
1961.KL       0.067270     0.171527           0.024705
1818.KL       0.136962     0.182821           0.047193
1295.KL       0.128504     0.184452           0.043887
5347.KL       0.159319     0.185774           0.054023

HIGH VOLATILITY STOCKS (Stage 2 Addition)
         Annual Return  Annual Risk  Volatility  Mean Return  Meets_Condition
Ticker                                                                       
5168.KL      -0.288366     0.575177    0.036233    -0.001144             True
5216.KL      -0.011250     0.5

## 9. Data Export

Save datasets for reproducibility and portfolio optimisation stage.

In [11]:
os.makedirs("output", exist_ok=True)

# Check if output already exists
if not os.path.exists("output/selected_stocks.csv"):
    print("📁 Saving outputs (first run)...")
    
    os.makedirs("output", exist_ok=True)
    
    # Save candidate pool
    pd.Series(stocks, name="Candidate Pool").to_csv("output/candidate_pool.csv")
    print(f"Candidate pool size: {len(stocks)} stocks")
    
    # Save processed datasets
    stock_prices.to_csv("output/cleaned_prices.csv")
    returns.to_csv("output/returns.csv")
    stats_full.to_csv("output/stock_statistics.csv")
    
    # Save selected stocks
    low_vol.to_csv("output/low_vol_stocks.csv")
    high_vol.to_csv("output/high_vol_stocks.csv")
    
    # Combine selected stocks
    selected_stocks = list(low_vol.index) + list(high_vol.index)
    pd.Series(selected_stocks, name="Selected Stocks").to_csv("output/selected_stocks.csv")
    
    print("\n✅ Core outputs saved")
else:
    print("ℹ️ Output files already exist. Skipping save. Delete 'output' folder to re-save.")

Candidate pool size: 40 stocks

✅ Core outputs saved


## 10. Raw Data Export (Optional)

Run only once for dataset generation.

In [13]:
SAVE_DATA = False  # Change to True ONLY when exporting

if SAVE_DATA:

    print("Saving raw datasets...")

    os.makedirs("raw_data", exist_ok=True)

    for ticker in stock_prices.columns:
        df = stock_prices[[ticker]].dropna()
        df.to_csv(f"raw_data/{ticker}.csv")

    benchmark_prices.to_csv("raw_data/fbmklci.csv")

    print("✅ Raw data saved")

else:
    print("ℹ️ Raw data export skipped")

Saving raw datasets...
✅ Raw data saved
